# Frontier Workload Statistical Characterization (single day: 2024-04-07)

**Rationale.** Statistical analysis of Frontier workloads/compute-power tracing before building a synthetic compute-power generator. By characcterizing the real measured workload we are able to build the framework for a power-trace data synthesizer loyal to the Frontier HPC's realistic deployment situations & physical constraints. To support regime A parameters & regime B/C...n distribution shifts.

**Data Source.** `input_04-07-24.csv` - 25 per-rack power columns + OA wet-bulb, 5761 rows at 15 s = 24 h — real measured granularity. Sustain-lc implements a smoothing disaggregator algorithm that:
1. Divides power tracing by 5 (an interjected scalar, author noted "???" after their decision, keep constant for benchmarking purposes.)
2. Then divides by 3 (to artificially make the data more granular to the blade group level.)
3. repeat(3) to spread per-rack power to per-bladegroup.
4. phase-shifts branch 2 & 3's data by 7.5 & 15 hrs, respectively, to artificially create temporal shifts/differences.
5. stacks cols 6-10 after 1-5, then after that cols 11-15, repeat til column 75 soi there are 15 columns, one for each blade group.
6. softmax & smooth, again for artifical structure & controls.

^^^when we design our own disaggregator algorithm, ensure that there is not so much built-in temporal & spatial structure. (steps 4 & 6).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, pathlib

# data lives in the sustain-lc submodule (walk up from cwd to find it; move-safe)
SUSTAIN_LC = next(p / "external" / "sustain-lc"
                  for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                  if (p / "external" / "sustain-lc").is_dir())
sys.path.insert(0, str(SUSTAIN_LC))

df = pd.read_csv(SUSTAIN_LC / 'input_04-07-24.csv')

POWER = [c for c in df.columns if c.startswith('power')]   # 25 per-rack columns
t_h = df['time'].values / 3600.0                            # time axis in hours
P = df[POWER].values / 1e6                                  # rack power in MW, shape (T, 25)
total = P.sum(axis=1)                                       # facility compute power in MW
dt = np.unique(np.diff(df['time'].values))

print(f'rows={len(df)}  racks={len(POWER)}  span={t_h[-1]:.1f} h  dt={dt} s')

## 1. Per-rack marginals

How much power does each rack draw, and are the 25 racks interchangeable or structurally different? This decides whether the generator can treat racks as exchangeable (one shared marginal + a per-rack scale) or must carry 25 separate distributions.

**Decision gate.** If means cluster tightly with one or two outliers (watch rack 14), racks are *nearly exchangeable*: the generator needs one shared marginal shape plus a small per-rack scale factor.

In [ ]:
stats = pd.DataFrame({
    'mean_MW': P.mean(0), 'std_MW': P.std(0),
    'min_MW': P.min(0), 'max_MW': P.max(0),
    'p05_MW': np.percentile(P, 5, 0), 'p95_MW': np.percentile(P, 95, 0),
}, index=[f'r{i+1}' for i in range(P.shape[1])]).round(3)
lo, hi = stats['mean_MW'].min(), stats['mean_MW'].max()
print(f'across-rack spread of the per-rack mean: {lo:.3f} - {hi:.3f} MW')
display(stats)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(1, 26), stats['mean_MW'], yerr=stats['std_MW'], capsize=2, color='steelblue')
ax.set_xlabel('rack'); ax.set_ylabel('mean power (MW)')
ax.set_title('Per-rack mean power (error bars = std over the day)')
ax.set_xticks(range(1, 26))
plt.tight_layout(); plt.show()

## 2. Temporal structure

Is the day smooth (slow diurnal drift) or step-like (discrete job boundaries)? Step structure argues for a **job-superposition** generator; smooth structure means a statistical model is enough.

**Decision gate.** A heavy-tailed ramp distribution (most steps ≈ 0, rare large jumps — quantified below by *excess kurtosis*) = discrete job arrivals/departures on top of a slow envelope → job-superposition is the physically honest generator and gives an interpretable regime-B knob ("bigger / longer / burstier jobs"). A long autocorrelation means power persists for many steps, so the generator must model job *duration*, not just per-step noise.

**Caveat (parallels §3).** The autocorrelation below is computed on the *total* signal, which is dominated by the diurnal envelope. So the reported decorrelation time is an **upper bound inflated by the day's trend** — it is *not* a clean measure of per-job duration. Cleanly isolating job persistence would need per-rack, detrended autocorrelation; the aggregate number here only establishes that the signal is far from white noise.

In [ ]:
# All 25 traces + the facility total
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t_h, P, lw=0.5, alpha=0.35)
ax[0].set_ylabel('per-rack MW'); ax[0].set_title('All 25 rack traces')
ax[1].plot(t_h, total, color='k', lw=1.2)
ax[1].set_ylabel('total MW'); ax[1].set_xlabel('hour of day')
ax[1].set_title(f'Facility compute power  (range {total.min():.1f} - {total.max():.1f} MW)')
ax[1].set_xlim(0, 24)
plt.tight_layout(); plt.show()

# Ramp magnitude per 15 s step, and autocorrelation of the total signal
ramps = np.diff(total)
amean, ap99, amax = np.abs(ramps).mean(), np.percentile(np.abs(ramps), 99), np.abs(ramps).max()
print(f'ramp |dP| per 15 s step:  mean={amean:.3f}  p99={ap99:.3f}  max={amax:.3f} MW')

# Quantify "heavy-tailed" instead of eyeballing: excess kurtosis of the ramp distribution.
zr = (ramps - ramps.mean()) / ramps.std()
print(f'ramp excess kurtosis = {(zr**4).mean() - 3:.1f}  (0 = Gaussian; large positive => heavy tails / rare large jumps = discrete job boundaries)')

x = total - total.mean()
ac = np.correlate(x, x, 'full')[len(x) - 1:]
ac = ac / ac[0]
lag_min = np.arange(len(ac)) * 15 / 60.0

# Quantify "long autocorrelation": first lag where ac drops below 1/e.
# NB: computed on the TOTAL signal -> inflated by the diurnal trend, so this is an
# UPPER bound on job persistence, not a measure of per-job duration (see markdown caveat).
tau = int(np.argmax(ac < 1 / np.e))
print(f'autocorr 1/e decorrelation time = {tau} steps = {tau * 15 / 60:.0f} min  (trend-inflated upper bound, not pure job duration)')

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(ramps, bins=80, color='darkorange'); ax[0].set_yscale('log')
ax[0].set_title('Step-to-step change in total'); ax[0].set_xlabel('dP (MW/step)')
ax[1].plot(lag_min[:240], ac[:240]); ax[1].axhline(0, color='grey', lw=0.5)
ax[1].axhline(1 / np.e, color='crimson', lw=0.5, ls='--')
ax[1].set_title('Autocorrelation of total power'); ax[1].set_xlabel('lag (min)')
plt.tight_layout(); plt.show()

## 3. Cross-rack correlation - the headline

This is the highest-leverage quantity. Frontier gang-schedules jobs across racks, so rack powers are correlated - and that correlation is exactly what the GNN keys on and what one real day under-determines. We look two ways, because raw correlation is inflated by the shared diurnal trend:

- **(a) raw correlation** - the 25x25 matrix and its off-diagonal distribution;
- **(b) PC1 removed** - factor out the dominant shared component and inspect the *residual* correlation. This separates "all racks follow one global signal" (rank-1) from genuine independent / block structure.

**Decision gate.** If PC1 explains almost everything (near rank-1), the day is `global_signal(t) * scale_r + small noise`: the generator factorizes that way and the regime-B knob is mostly reshaping the global signal + noise floor - **but for the GNN this is a warning**: near-redundant rack nodes let message passing collapse to pooling. If the *residual* map shows block structure, those blocks are genuine rack communities -> candidate GNN edges and a group-level job model.

In [ ]:
iu = np.triu_indices(25, 1)

# (a) Raw correlation
C = np.corrcoef(P.T)
print(f'raw off-diagonal corr:  mean={C[iu].mean():.3f}  min={C[iu].min():.3f}  max={C[iu].max():.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(C, vmin=0, vmax=1, cmap='viridis')
ax[0].set_title('Raw rack-rack correlation'); fig.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].hist(C[iu], bins=40, color='seagreen')
ax[1].set_title('Off-diagonal correlation histogram'); ax[1].set_xlabel('corr')
plt.tight_layout(); plt.show()

# (b) Is the day essentially ONE global signal scaled per rack (rank-1)?
# PCA on the standardized racks. Try eigh; fall back to power iteration
# (pure matmul, no LAPACK) so this cell can never hard-crash the kernel.
Z = (P - P.mean(0)) / P.std(0)
R = np.cov(Z.T)                       # 25 x 25
try:
    w, V = np.linalg.eigh(R)
    w, V = w[::-1], V[:, ::-1]        # descending
    ve = w / w.sum()
    v1 = V[:, 0]
except Exception as e:
    print('eigh failed (%s); using power iteration for PC1 only' % type(e).__name__)
    v = np.ones(R.shape[0])
    for _ in range(1000):
        Rv = R @ v; v = Rv / np.sqrt((Rv * Rv).sum())
    lam1 = float(v @ R @ v); v1 = v
    ve = np.array([lam1 / np.trace(R)])

print('variance explained by PC1..PC5:', np.round(ve[:5], 4))

resid = Z - np.outer(Z @ v1, v1)      # remove the dominant shared component
Cr = np.corrcoef(resid.T)
print(f'residual off-diag corr (PC1 removed):  mean={Cr[iu].mean():.3f}  std={Cr[iu].std():.3f}  absmax={np.abs(Cr[iu]).max():.3f}')

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(Cr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_title('Residual corr after removing PC1'); fig.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.show()

## 4. Aggregate vs. the year envelope

Does this one day cover the operating range, or is it a narrow slice? The full-year 2023 xlsx spans ~11-26 MW total compute. We check where this day sits so regime A is calibrated to a representative load and we know how far regime B must push to be genuinely out-of-distribution.

**Decision gate.** If the day already sweeps a wide band, it is a usable regime-A calibration day; regime B must then shift along an axis the day does *not* already cover (sustained high load, faster ramps, higher synchronization), and the A->B gap must be quantified to prove it is a real shift, not resampling noise.

In [ ]:
print(f'this day total compute: {total.min():.1f} - {total.max():.1f} MW  (mean {total.mean():.1f})')
print('year envelope (from the 2023 xlsx): ~11 - 26 MW total compute')

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(total, bins=50, color='slateblue')
ax[0].set_title('Distribution of total compute power'); ax[0].set_xlabel('MW')
ax[1].plot(t_h, total, color='slateblue'); ax[1].set_xlim(0, 24)
ax[1].set_title('Diurnal shape'); ax[1].set_xlabel('hour of day'); ax[1].set_ylabel('MW')
plt.tight_layout(); plt.show()

## 5. Wet-bulb (context only - sourced, not synthesized)

`Towb` is the *other* exogenous driver, but the plan is to **source** it from NOAA Oak Ridge records rather than synthesize it (one channel folds dry-bulb + humidity). Shown here only to see the day's range and to flag the +10/+15 K offset the env adds before feeding the FMU - that offset shifts the cooling-tower operating regime and will need a decision when real wet-bulb is fed in.

In [ ]:
wb = df['OA Wetbulb Temp'].values
print(f'OA wet-bulb: {wb.min():.2f} - {wb.max():.2f} C  (env adds +10 K in generators / +15 K in env before the FMU)')

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(t_h, wb, color='firebrick')
ax.set_xlim(0, 24)
ax.set_xlabel('hour of day'); ax.set_ylabel('wet-bulb (C)')
ax.set_title('OA wet-bulb over the day')
plt.tight_layout(); plt.show()

## 6. Freezing Analysis for Workload Synthesizer

In [ ]:
import json, pathlib

spec = {
    "meta": {
        "source_csv": "input_04-07-24.csv",
        "date": "2024-04-07",
        "n_racks": int(P.shape[1]),
        "dt_s": int(dt[0]),
        "n_steps": int(P.shape[0]),
        "units": "watts",
        "provenance": "frozen from workload_analysis.ipynb",
    },
    "marginal": {
        "grand_mean_W": float(P.mean() * 1e6),          # P is in MW in the nb -> back to W
        "per_rack_scale": (stats["mean_MW"] / stats["mean_MW"].mean()).tolist(),
        # ... idle_floor_W, busy_level_W, pooled_norm_quantiles_W, per_rack_CoV
    },
    "temporal": {
        "ramp_excess_kurtosis": float((zr**4).mean() - 3),
        # ... ramp_abs_mean_W, p99, max, autocorr_1e_tau_steps
    },
    "cross_rack": {
        "offdiag_corr_mean": float(C[iu].mean()),
        "pc1_var_share": float(ve[0]),
        "corr_matrix": C.tolist(),                       # 25x25 reference
        # ... min, max, residual_offdiag_absmax
    },
    "aggregate": { },   # total min/max/mean/std in W
    "tolerances": { },  # the acceptance bands we discussed
}

spec_path = SUSTAIN_LC.parents[1] / "workload_gen" / "spec" / "regime_A.json"
spec_path.parent.mkdir(parents=True, exist_ok=True)
spec_path.write_text(json.dumps(spec, indent=2))
print("froze", spec_path)


## 7. Summary/FIndings

All figures below are this single day (2024-04-07); they are *calibration targets*, not population truth (n = 1 day).

| Finding (this day) | Generator implication | GNN implication |
|---|---|---|
| Racks near-exchangeable (means ≈ 0.62–0.68 MW; rack 14 low at 0.42) | one shared marginal template + per-rack scale, not 25 independent dists | rack nodes share a feature template |
| Ramp distribution heavy-tailed (excess kurtosis ≈ 25); total-power autocorr long (1/e ≈ 44 min, **trend-inflated upper bound**) | job-superposition with explicit job *duration* | dynamics model must capture persistence |
| Raw cross-rack corr = **0.994**; PC1 = **99.4%** of variance (essentially rank-1) | factorized `global(t)·scale_r + eps`; shift knob = reshape global signal + noise floor | **risk:** near-redundant nodes → message passing may collapse to pooling (the star-topology / DeepSets risk) |
| Residual corr after PC1 removal **collapses to noise** (mean ≈ −0.04, \|max\| ≈ 0.42) → **no rack-community block structure on this day** | no group structure to *learn* from one day; cross-rack synchronization must be supplied by the generator's job-placement model, not fit from this trace | no community edges learnable from this day → the DeepSets-collapse risk is concrete, not hypothetical |
| Day spans ≈ 5–27 MW (mean 16) | wide enough to be a usable regime-A calibration day | – |

**Caveats carried forward.**
- One day = one autocorrelated sample. These statistics pin regime A's *targets* but cannot teach a *distribution over* job patterns — that needs job-level trace data (PM100/PWA/F-DATA).
- The 25 columns are aggregate per-rack power; individual jobs (arrivals/sizes/durations) are *not* recoverable from this sum. The generator fits job stats from external HPC logs and is *calibrated* to match these aggregate targets (forward-fit, not inversion).

**Data Sourcing & Build Design Directions**
1. Workload/Power trace synthesizer is built from the spatial & temporal distribution modelled off of the one-day granular whole-Frontier power tracing.
2. Enveloped with the year-round data to extend into summer Vs. winter shifts/seasonal shifts.
3. Source weather data from NOAA near Oakridge for weather shits & match with envelope.
4. Extend synthesizer (shape jobs characteristics/types) with other open-sourced HPC data.